# Multi-Entity Standardization Notebook (Bronze → Silver)

This Databricks notebook implements a **config-driven**, **semantic-based** standardization framework for processing one source into multiple silver entities.


In [ ]:
# Databricks runtime parameter
try:
    dbutils.widgets.text("source_name", "")
except NameError:
    # Allows local dry-run without Databricks widgets
    pass

source_name = dbutils.widgets.get("source_name") if "dbutils" in globals() else ""
if not source_name:
    raise ValueError("Input parameter 'source_name' is required.")

import json
import uuid
from typing import Dict, List, Tuple

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


In [ ]:
# -----------------------------
# Helper functions (modular)
# -----------------------------

def clean_strings(df: DataFrame, data_types: Dict[str, str]) -> DataFrame:
    """Trim/lower/remove unwanted chars for all configured string columns."""
    for col_name, dtype in (data_types or {}).items():
        if col_name in df.columns and str(dtype).lower() in {"string", "str", "varchar"}:
            df = df.withColumn(
                col_name,
                F.regexp_replace(F.lower(F.trim(F.col(col_name))), r"[^a-z0-9_\s@.\-]", "")
            )
    return df


def apply_semantic_rules(
    df: DataFrame,
    semantic_types: Dict[str, str],
    source_timezone: str = "UTC"
) -> DataFrame:
    """Apply semantic-driven transformations without hardcoded column names."""
    for col_name, semantic in (semantic_types or {}).items():
        if col_name not in df.columns:
            continue

        semantic_lc = str(semantic).lower()

        if semantic_lc == "phone":
            df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"[^0-9]", ""))

        elif semantic_lc == "postal":
            df = df.withColumn(col_name, F.regexp_replace(F.upper(F.col(col_name)), r"\s+", ""))

        elif semantic_lc == "date":
            ts_col = F.to_timestamp(F.col(col_name))
            if source_timezone and source_timezone.upper() != "UTC":
                ts_col = F.to_utc_timestamp(ts_col, source_timezone)
            df = df.withColumn(col_name, ts_col)

        elif semantic_lc == "currency":
            df = df.withColumn(
                col_name,
                F.regexp_replace(F.regexp_replace(F.col(col_name), r"\$", ""), r",", "").cast("double")
            )

        elif semantic_lc == "boolean":
            normalized = F.lower(F.trim(F.col(col_name).cast("string")))
            df = df.withColumn(
                col_name,
                F.when(normalized.isin("true", "t", "yes", "y", "1"), F.lit(True))
                 .when(normalized.isin("false", "f", "no", "n", "0"), F.lit(False))
                 .otherwise(F.lit(None).cast("boolean"))
            )

    return df


def add_lineage(df: DataFrame, source_name: str, run_id: str) -> DataFrame:
    return (
        df.withColumn("source_name", F.lit(source_name))
          .withColumn("ingestion_timestamp", F.current_timestamp())
          .withColumn("run_id", F.lit(run_id))
    )


def deduplicate(df: DataFrame, primary_keys: List[str]) -> DataFrame:
    usable_keys = [k for k in (primary_keys or []) if k in df.columns]
    return df.dropDuplicates(usable_keys) if usable_keys else df


In [ ]:
def load_active_config(source_name: str) -> Dict[str, str]:
    cfg_df = (
        spark.table("config.control_table")
             .filter((F.col("source_name") == source_name) & (F.col("active_flag") == F.lit(True)))
             .limit(1)
    )

    rows = cfg_df.collect()
    if not rows:
        raise ValueError(f"No active config found in config.control_table for source_name={source_name}")

    cfg = rows[0].asDict(recursive=True)
    if not cfg.get("mapping_config_path"):
        raise ValueError("mapping_config_path is missing in config.control_table")

    return cfg


def read_bronze_data(source_name: str) -> DataFrame:
    return spark.table(f"bronze.{source_name}")


def list_entity_config_files(mapping_config_path: str) -> List[str]:
    files = dbutils.fs.ls(mapping_config_path)
    return [f.path for f in files if f.path.lower().endswith(".json")]


def load_entity_config(config_file_path: str) -> Dict:
    raw_json = dbutils.fs.head(config_file_path, 10 * 1024 * 1024)
    entity_cfg = json.loads(raw_json)

    required_sections = [
        "entity_name",
        "column_mappings",
        "data_types",
        "default_values",
        "primary_keys",
        "semantic_types",
    ]
    missing_sections = [k for k in required_sections if k not in entity_cfg]
    if missing_sections:
        raise ValueError(f"Missing keys in {config_file_path}: {missing_sections}")

    return entity_cfg


def process_entity(
    df_bronze: DataFrame,
    source_name: str,
    run_id: str,
    entity_cfg: Dict,
    source_timezone: str = "UTC"
) -> Tuple[str, int]:
    entity_name = entity_cfg["entity_name"]
    column_mappings = entity_cfg.get("column_mappings", {})
    data_types = entity_cfg.get("data_types", {})
    default_values = entity_cfg.get("default_values", {})
    primary_keys = entity_cfg.get("primary_keys", [])
    semantic_types = entity_cfg.get("semantic_types", {})

    # 4.2 Select only required source columns (entity isolation)
    required_source_cols = list(column_mappings.keys())
    existing_source_cols = [c for c in required_source_cols if c in df_bronze.columns]
    if not existing_source_cols:
        raise ValueError(f"No mapped source columns found in bronze data for entity={entity_name}")

    df_entity = df_bronze.select(*existing_source_cols)

    # 4.3 Apply column mapping (source -> target)
    for src_col, tgt_col in column_mappings.items():
        if src_col in df_entity.columns:
            df_entity = df_entity.withColumnRenamed(src_col, tgt_col)

    # 4.4 Add missing target columns with null
    target_cols = set(column_mappings.values()) | set(data_types.keys()) | set(default_values.keys())
    for tgt_col in target_cols:
        if tgt_col not in df_entity.columns:
            df_entity = df_entity.withColumn(tgt_col, F.lit(None))

    # 4.5 Default values
    if default_values:
        df_entity = df_entity.fillna(default_values)

    # 4.6 Data type casting
    for col_name, dtype in data_types.items():
        if col_name in df_entity.columns:
            df_entity = df_entity.withColumn(col_name, F.col(col_name).cast(dtype))

    # 4.7 Standardization rules
    df_entity = clean_strings(df_entity, data_types)
    df_entity = apply_semantic_rules(df_entity, semantic_types, source_timezone=source_timezone)

    # Re-cast after semantic standardization (e.g., currency/date conversions)
    for col_name, dtype in data_types.items():
        if col_name in df_entity.columns:
            df_entity = df_entity.withColumn(col_name, F.col(col_name).cast(dtype))

    # 4.8 Lineage columns
    df_entity = add_lineage(df_entity, source_name=source_name, run_id=run_id)

    # 4.9 Deduplication
    df_entity = deduplicate(df_entity, primary_keys=primary_keys)

    # 4.10 Write Silver Delta
    silver_table = f"silver.{entity_name}"
    (
        df_entity.write
                 .format("delta")
                 .mode("append")
                 .saveAsTable(silver_table)
    )

    # 4.11 Logging
    record_count = df_entity.count()
    print(f"[SUCCESS] entity={entity_name}, records={record_count}, target={silver_table}")

    return entity_name, record_count


In [ ]:
# -----------------------------
# Driver logic (multi-entity loop)
# -----------------------------

run_id = str(uuid.uuid4())

cfg = load_active_config(source_name)
mapping_config_path = cfg["mapping_config_path"]
source_timezone = cfg.get("source_timezone", "UTC")

print(f"Starting standardization | source_name={source_name} | run_id={run_id}")
print(f"mapping_config_path={mapping_config_path}")

df_bronze = read_bronze_data(source_name)
entity_config_files = list_entity_config_files(mapping_config_path)

if not entity_config_files:
    raise ValueError(f"No JSON entity config files found under {mapping_config_path}")

summary = []
errors = []

for config_file in entity_config_files:
    try:
        entity_cfg = load_entity_config(config_file)
        entity_name, record_count = process_entity(
            df_bronze=df_bronze,
            source_name=source_name,
            run_id=run_id,
            entity_cfg=entity_cfg,
            source_timezone=source_timezone,
        )
        summary.append({"entity_name": entity_name, "record_count": record_count, "status": "SUCCESS"})
    except Exception as ex:
        print(f"[ERROR] config_file={config_file} | error={str(ex)}")
        errors.append({"config_file": config_file, "error": str(ex), "status": "FAILED"})

print("\n=== Run Summary ===")
for row in summary:
    print(row)

if errors:
    print("\n=== Entity Failures (continued processing) ===")
    for err in errors:
        print(err)
